# 00 · Introduction & Setup
## AFM Image Analysis with Python Leeds AFM & SPM Workshop 2026

Welcome! This workshop covers:

- loading AFM data
- processing pipelines for AFM images
- analysis using Python
- exporting publication-quality results

This notebook assumes you've already followed the **Setup Instructions** in the repository's `README.md` — Miniforge installed, the repo cloned, the `leeds_python_workshop` conda environment created and activated, and Jupyter launched from inside the repo folder. If you haven't done that yet, do it now; the checks below will tell you quickly if something didn't go to plan.

This notebook does three things:

1. Gets you oriented in **Jupyter**, if you haven't used it before.
2. **Checks your environment** is correctly set up.
3. Gives a **60-second tour** of the tools you'll meet over the next four notebooks.

**Contents**
1. Using Jupyter notebooks
2. Environment check
3. A tour of the tools
   - NumPy and arrays
   - AFMReader
   - SciPy, scikit-image, scikit-learn
   - Other AFM software (not used today)
4. What's next

---

## 1 · Using Jupyter notebooks

If you've used Jupyter before, skip ahead to Section 2.

A notebook is a sequence of **cells**. There are two kinds:

- **Markdown cells** (like this one): formatted text, for explanation.
- **Code cells**: runnable Python.

A few essentials:

- **Run a cell**: click it, then press `Shift + Enter` (runs it and moves to the next
  cell) or `Ctrl + Enter` (runs it and stays put).
- **Order matters, not position.** Code only "exists" once its cell has been run. If
  you skip a cell, anything it defines doesn't exist yet — you'll get a
  `NameError`. Run cells top to bottom, in order, especially the first time through a
  notebook.
- **The `[ ]` next to a code cell** shows its status: empty means not yet run, `[*]`
  means currently running, `[6]` means it was the 6th cell run *this session* (not
  necessarily the 6th cell in the notebook — that number tells you the run order, which
  is useful for spotting cells that got run out of sequence).
- **If something seems stuck or wrong**: use the Kernel menu (or the restart button —
  often a circular arrow icon in the toolbar) to *Restart Kernel*, then *Run All* from
  the top. This clears everything and starts fresh — the single most useful fix for
  "this used to work" problems.
- **The kernel** is the Python process actually running your code. Its name (shown top
  right, or in a Kernel menu depending on your setup) should match the
  `leeds_python_workshop` environment from the README — Section 2 checks this directly.

Try it now:

In [ ]:
# Click this cell, then press Shift+Enter
print("If you can see this, you're running cells correctly!")

That's genuinely most of what you need. Everything else (closing/reopening, saving, etc.) works like any other programme.

> TIP: The double trangle (⏩) button at the top lets you restart and run ALL the cells from the top.

## 2 · Environment check

Run the cells below in order. **They're designed to tell you exactly what's wrong if
something is wrong** — read the printed output rather than just a traceback, if there
is one.

If anything here doesn't pass, flag it now rather than during a later notebook; it's
much easier to fix environment problems before we're relying on the environment for
real work.

### 2.1 · Are you in the right conda environment?

The README has you create and activate an environment called `leeds_python_workshop`.
If Jupyter is using a different kernel — your base conda environment, a different
project's environment, whatever was previously selected — none of the installed
packages will be visible, even if the install itself worked perfectly.

In [ ]:
import os
import sys

conda_env = os.environ.get("CONDA_DEFAULT_ENV")
print(f"Python executable: {sys.executable}")
print(f"Active conda environment: {conda_env!r}")
print()

if conda_env == "leeds_python_workshop":
    print("Correct environment is active.")
else:
    print("This doesn't look like the workshop environment.")
    print("Fix: in the Kernel menu, choose 'Change Kernel' and select")
    print("'leeds_python_workshop' (or close to that name). If it isn't")
    print("listed at all, the environment may not have installed correctly —")
    print("see the README's Troubleshooting section.")

### 2.2 · Are the required packages installed?

This checks every package the workshop notebooks import, and notes which conda
environment.yml section each comes from (so a gap is easy to trace back).

In [ ]:
import importlib

# (package import name, human label, where it comes from in environment.yml)
required_packages = [
    ("numpy",      "numpy",        "conda"),
    ("matplotlib", "matplotlib",   "conda"),
    ("scipy",      "scipy",        "conda"),
    ("skimage",    "scikit-image", "conda"),
    ("pandas",     "pandas",       "conda"),
    ("sklearn",    "scikit-learn", "conda"),
    ("tifffile",   "tifffile",     "conda"),
    ("PIL",        "pillow",       "conda"),
    ("AFMReader",  "AFMReader",    "pip"),
]

missing = []
for import_name, label, source in required_packages:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "version unknown")
        print(f"  - {label:14s} {version:14s} ({source})")
    except ImportError:
        print(f"  X {label:14s} NOT FOUND        ({source})")
        missing.append(label)

print()
if missing:
    print(f"MISSING: {', '.join(missing)}")
    print()
    print("If you're confident you're in the right environment (Section 2.1 above")
    print("passed) and packages are still missing, the environment.yml itself may")
    print("be missing them as explicit dependencies — flag this to the organisers.")
else:
    print("All required packages found.")

### 2.3 · Finding the workshop data

All of today's notebooks (including this one) sit at the top level of the repo, alongside `data/`, `resources/`, and
`environment.yml`, so `Path(".")` correctly points at the repo root.

The cell below confirms the sample data and colormap files are actually where they're expected to be. The single most common cause of a "file not found" error later in the workshop isn't a code bug, it's a notebook being opened from somewhere other than the
repo's top level.

In [ ]:
from pathlib import Path

REPO_ROOT = Path(".")
DATA_DIR = REPO_ROOT / "data" / "samples"
COLORMAP_DIR = REPO_ROOT / "resources" / "colormaps"

print(f"This notebook's directory: {Path.cwd()}")
print(f"Detected repo root:        {REPO_ROOT}")
print()

print(f"Looking for sample data in: {DATA_DIR}")
if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
    print(f"  Found {len(list(DATA_DIR.iterdir()))} files.")
else:
    print("  Not found, or empty.")

print()
print(f"Looking for colormap files in: {COLORMAP_DIR}")
if COLORMAP_DIR.exists():
    csv_files = sorted(p.name for p in COLORMAP_DIR.glob("*.csv"))
    print(f"  Found: {', '.join(csv_files) if csv_files else '(none)'}")
else:
    print("  Not found.")

> 📌 **Note:** notebooks 01–04 currently use
> `DATA_DIR = Path(".") / "data" / "samples"`, which is correct *as long as those
> notebooks stay at the repo's top level* alongside this one. If they're ever moved
> into a subfolder (e.g. to match a `notebooks/` structure some READMEs describe),
> those paths would need updating too — either to `Path("..")`, or by reusing the
> `find_repo_root()` helper above. Not an issue today, just worth knowing if the repo
> layout changes later.

### 2.4 · Registering the workshop's AFM colormaps

The same registration function used in every other notebook, confirmed here first.
This version reports a problem with **one** colormap file without crashing the whole
check, so you can see at a glance which (if any) are missing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def register_afm_colormaps(folder=COLORMAP_DIR):
    """Load the workshop's AFM colormaps from CSV and register them with matplotlib.
    Safe to call more than once — already-registered colormaps are skipped."""
    names = ("afm_brown", "playnano_gold", "classic_afm", "aurea")
    ok, failed = [], []
    for name in names:
        if name in plt.colormaps():
            ok.append(name)
            continue
        try:
            rgb_data = np.loadtxt(folder / f"{name}.csv", delimiter=",")
            cmap = ListedColormap(rgb_data, name=name)
            plt.colormaps.register(cmap)
            plt.colormaps.register(cmap.reversed(), name=f"{name}_r")
            ok.append(name)
        except FileNotFoundError:
            failed.append(name)
    return ok, failed

ok, failed = register_afm_colormaps()
print(f"Registered: {', '.join(ok) if ok else '(none)'}")
if failed:
    print(f"Could not find: {', '.join(failed)} -- check resources/colormaps/ for these CSVs.")
else:
    print("All colormaps registered successfully.")

## 3 · A tour of the tools

A quick orientation, not a tutorial. Each of these gets used properly, with real
examples, in the notebooks that follow; this is just so the names aren't new when you
meet them there.

### 3.1 · NumPy and arrays

Almost everything in this workshop is a **NumPy array** under the hood. An AFM image is just a 2D grid of height values; an AFM video is a 3D stack of those grids. Once your data is a NumPy array, every tool in this workshop, and most of scientific Python, can work with it.

A few core ideas, with a tiny made-up "image" to try them on:

In [ ]:
# A small 5x5 array standing in for a tiny AFM image (real ones are much bigger!)
tiny_image = np.array([
    [0.1, 0.2, 0.1, 0.0, 0.1],
    [0.2, 0.5, 0.9, 0.4, 0.2],
    [0.1, 0.8, 1.5, 0.7, 0.1],
    [0.2, 0.4, 0.8, 0.3, 0.2],
    [0.1, 0.1, 0.2, 0.1, 0.0],
])

print("Shape:", tiny_image.shape, " (rows, columns)")
print("Dtype:", tiny_image.dtype)
print()
print("Whole array:\n", tiny_image)

In [ ]:
# Indexing: [row, column]. Slices work just like Python lists, in both dimensions.
print("Single pixel, row 2 col 2:", tiny_image[2, 2])
print("Whole row 2:               ", tiny_image[2, :])
print("Whole column 2:            ", tiny_image[:, 2])
print("Top-left 2x2 corner:\n", tiny_image[:2, :2])

In [ ]:
# Arithmetic applies to every element at once -- no manual loop needed.
# This is the exact operation "levelling" relies on in notebook 02: subtract a
# reference value from every pixel.
mean_height = tiny_image.mean()
print(f"Mean height: {mean_height:.3f}")

centred = tiny_image - mean_height
print("After subtracting the mean (now centred on 0):\n", centred.round(3))

That's genuinely most of the NumPy this workshop relies on: shape, indexing, and
whole-array arithmetic. Everything else — fitting, filtering, segmenting — is built on
exactly these operations, including the batch and video-processing pipelines later on,
where the same operations simply apply to a 3D stack instead of a single 2D image.

### 3.2 · AFMReader

[**AFMReader**](https://afm-spm.github.io/AFMReader/) is a library with one job:
turning AFM files from many different microscope manufacturers (Bruker `.spm`,
Asylum `.ibw`, JPK `.jpk`/`.h5-jpk`, and more) into plain NumPy arrays, so you never
have to write a file parser yourself. It's the one workshop dependency installed via
`pip` rather than `conda` (see `environment.yml`).

Notebook 01 covers this properly, for now, just know that nearly every notebook starts with a line that looks like:

```python
image, pixel_to_nm = LoadFile(file_path, channel="...").load()
```

and from that point on, `image` is just a NumPy array like the one above.

### 3.3 · SciPy, scikit-image, and scikit-learn

Three general-purpose scientific Python libraries do almost all the heavy lifting in
notebooks 02–04. You don't need to know their APIs yet, just what role each plays:

- **`scikit-learn`** (`sklearn`): best known for machine learning, but here we only use
  its straightforward *regression* tools (`LinearRegression`, `PolynomialFeatures`) to
  fit planes and curved surfaces to AFM data, for levelling. (Notebook 02.)
- **`scikit-image`** (`skimage`): an image processing library that helps with thresholding (`filters`),
   labelling and measuring connected regions (`measure`), removing edge-touching objects
  (`segmentation`), and colour-related tools (`color`). This is the main library behind
  segmenting and measuring individual molecules or features; the core of building a
  processing pipeline. (Notebooks 02–04.)
- **`scipy`**: a huge general scientific library; we use two small corners of it;
  `scipy.stats` for fitting distributions, and `scipy.ndimage` for Gaussian smoothing. (Notebooks 03–04.)

All three are mature, widely used, well-documented libraries; knowing they exist and roughly what they're for will help you recognise them (and search their docs confidently) long after this workshop.

### 3.4 · Other AFM software (not used in this workshop)

Python isn't the only way to do any of this — and it's worth knowing what else exists,
even though we won't use these directly today:

- **[playNano](https://github.com/derollins/playNano)**: a Python package for loading,processing, and exporting AFM images and high-speed AFM   videos (including GIF/video export), with the same kind of native colormaps used in this workshop.  Desinged to handle time-series and high   -speed AFM data. 
- **[TopoStats](https://github.com/AFM-SPM/TopoStats)**: an actively maintained command-line tool for *batch processing* AFM images:
  flattening, grain detection, and statistics, plus more advanced features like molecule tracing. Where this workshop builds small pieces
  by hand so you understand what's happening underneath, TopoStats is the tool you'd reach for once you want to process hundreds of images
  unattended with a settled, validated pipeline. It's built on the same AFMReader library notebook 01 uses.

Today's notebooks are deliberately "from scratch" — the goal is to understand what
levelling, masking, and measuring actually *do*, so that tools like these (or ones you
write yourself later) aren't a black box.

## 4 · What's next

If Sections 2.1–2.4 above all passed cleanly, you're ready. Open **`01_loading.ipynb`**
(in the same folder as this notebook) to get started — we'll load a real AFM image and
a real high-speed AFM video, and see what they look like once they're NumPy arrays.
From there, work through in order:

1. `01_loading.ipynb`
2. `02_processing.ipynb`
3. `03_analysis.ipynb`
4. `04_export.ipynb`

If anything in Section 2 didn't pass, don't worry; flag it now and we'll help
troubleshoot before we get going.